In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import Tool
from langchain.agents import create_agent
from langchain_tavily import TavilySearch

C:\Anaconda3\envs\langchain\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
load_dotenv(dotenv_path=".env")

api_key = os.getenv("OPENAI_API_KEY")

print(type(api_key))
print(api_key is not None)

tavily_key=os.getenv("TAVILY_API_KEY")
print(type(tavily_key))
print(tavily_key is not None)


<class 'str'>
True
<class 'str'>
True


In [4]:
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    api_key=api_key)

# Tool 1 : Simple QA

qa_prompt = ChatPromptTemplate.from_template(
    "Answer clearly: {question}"
)

qa_chain = qa_prompt | llm

qa_tool = Tool(
    name="Simple_QA",
    func=lambda question: qa_chain.invoke(
        {"question": question}
    ).content,
    description="Answers factual questions clearly."
)

# Tool 2 : Summarizer

summary_prompt = ChatPromptTemplate.from_template(
    "Summarize this text:\n\n{text}"
)

summary_chain = summary_prompt | llm

summary_tool = Tool(
    name="Summarizer",
    func=lambda text: summary_chain.invoke(
        {"text": text}
    ).content,
    description="Summarizes long paragraphs or text."
)

# Tool 3 : Tavily Web Search

search_tool = TavilySearch(
    max_results=3,
    tavily_api_key=tavily_key
)

#create Agent
tools=[
    qa_tool,
    summary_tool,
    search_tool]

agent=create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
"You are a helpful AI assistant."
"Use the appropriate tool whenever it improves the answer."
    )
)

#Run queries

queries=["What is LangGraph in LangChain?"]

for query in queries:
    print("\n🧑‍💻 User Query:", query)
    response=agent.invoke(
        {"messages":[{"role":"user", "content":query}]})

print("Action: simple_qa")
print("Action Input:", query)
print("\n🤖 Agent Response:\n")
print(response["messages"][-1].content)



🧑‍💻 User Query: What is LangGraph in LangChain?
Action: simple_qa
Action Input: What is LangGraph in LangChain?

🤖 Agent Response:

LangGraph in LangChain is a component designed to represent and manage complex chains of language model calls and their data dependencies using a graph structure. It enables visualization, organization, and execution of workflows where multiple language model prompts or chains are interconnected. This facilitates better understanding and control over the flow of data and operations within LangChain applications.
